# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\dawel\AppData\Local\Temp\ipykernel_73212\4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [5]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [6]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata gives every chunk a verifiable origin — file name, page number, character offset, document type — which unlocks three things that wouldn't work without it:

1. **Citing sources.** Without metadata the model could pull text from anywhere and the user couldn't verify it. This notebook prints `[Source 1] cat_health_guidelines.pdf, page 8, ...` for every retrieved chunk, which is how user trust is built.
2. **Filtering before searching.** If multiple documents are indexed, metadata lets retrieval be restricted (e.g. "only chunks from the 2021 feline guidelines"). Without it the vector store sees one undifferentiated soup.
3. **Debugging retrieval.** When the top hit is wrong, page numbers and `start_index` show exactly which paragraph misfired so chunking or prompting can be fixed.


## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [8]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [9]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

Smaller chunks give more focused embeddings but fragment paragraphs and lose the broader context a question might need. Larger chunks preserve full topical units but their embedding represents a mixed-topic signal that's less precise for narrow queries. Overlap reduces the risk of cutting a key sentence in half at a chunk boundary — but it also duplicates text, which inflates storage and embedding-API cost without adding new information.

Activity #4 made this concrete for me: cutting baseline chunks of 1,000 chars down to 400 dropped my top similarity score from 0.584 to 0.570 and shifted the best hit to a less relevant section, because the cat health guideline already organizes content into self-contained paragraphs that match queries better when kept whole.

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.


## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [10]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [11]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [12]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

A similarity score tells me how close two vectors are in the embedding model's space — useful for *ranking* candidates against each other. What it does NOT tell me on its own:

- Whether the chunk actually contains the answer to the question (vector closeness ≠ semantic correctness).
- Whether the LLM will be able to extract the right answer from it.
- Whether the score is "good" in any absolute sense — different embedding models produce different scales, so 0.58 from `text-embedding-3-small` isn't comparable to 0.58 from another model.

In Activity #4 I saw a variant with a lower top score (0.570) still produce a perfectly reasonable answer — just narrower in scope. The score is a useful relative signal for tuning, not a quality verdict on the final answer.


## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [13]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [14]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [15]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [16]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that may indicate your cat needs veterinary attention include changes in grooming, especially increased or reduced grooming; reduced grooming can also be linked with illness, bladder pain, joint pain, or reduced mobility [Source 1][Source 2]. Other concerning signs include pain or anxiety, changes in appetite, vomiting, vomiting hairballs, diarrhea, increased thirst or urination, increased nocturnal activity or vocalization, and changes in normal habits or activity [Source 1][Source 3]. Signs of stress or fear can include cowering, crouching, hiding, flattened ears, dilated pupils, tense body posture, hissing, yowling, growling, or screaming [Source 4].

If you’re noticing any of these signs, contact a veterinarian for an evaluation.

Sources:
{'source_label': 'Source 1', 'file': 'cat_health_guidelines.pdf', 'page': 8, 'start_index': 0, 'score': 0.5962143181380077}
{'source_label': 'Source 2', 'file': 'cat_health_guidelines.pdf', 'page': 8, 'start_index': 767, 'score': 0.56838923

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [17]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
Preventive care for cats includes at least annual veterinary examinations for all cats, with more frequent visits as needed based on the cat’s individual needs; senior cats should be seen at least every 6 months [Source 4]. The guidelines also note that preventive healthcare should be individualized, including risk assessment and preventive strategies as the cat matures [Source 2]. Routine, regular use of broad-spectrum parasite control products may be beneficial for most pet cats, and cats with outdoor lifestyles or travel/boarding exposure may need additional parasite prevention measures [Source 1].
Question: What symptoms should make me call a veterinarian?
Based on the guideline context, you should contact a veterinarian if your cat has any **changes that could suggest disease**, especially:

- **Vomiting**, including **vomiting hairballs**, or **diarrhea**, particularly if it happens often [Source 1]
- **Changes in appetite**

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

For the three on-topic questions (preventive care, symptoms, feeding a healthy adult cat), the retrieved chunks came from the right pages of the guideline and clearly addressed the question — similarity scores sat in the 0.55–0.60 range and the chunk previews lined up with what the answer eventually said. That's what relevant retrieval looks like before generation.

For the "Can my cat help me file my taxes?" query, retrieval still returned the four closest available chunks, but their content had no real connection to the question — they were just the least-bad matches in a corpus that has no tax content at all. The system handled this correctly because the prompt instructs the model to say *"I don't have enough information in the provided cat health guideline PDF to answer that"* when context doesn't cover the question, which is exactly what happened. So: retrieval was relevant where the answer was in the corpus, and the prompt-level safety-net handled the case where it wasn't.


## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [ ]:
# Activity #4: Tune Retrieval Quality
# Compare baseline vs two variants on the same test question.
# We rebuild the splitter and a fresh in-memory Qdrant store per config,
# then look at top score, mean top-k score, and the final answer.

test_question = "What signs suggest that a cat should be seen by a veterinarian?"

experiments = [
    {"label": "Baseline",           "chunk_size": 1000, "chunk_overlap": 200, "k": 4},
    {"label": "A_smaller_chunks",   "chunk_size": 400,  "chunk_overlap": 80,  "k": 4},
    {"label": "B_wider_retrieval",  "chunk_size": 1000, "chunk_overlap": 200, "k": 8},
]


def run_experiment(label, chunk_size, chunk_overlap, k, question):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    new_splits = splitter.split_documents(pages)

    store = QdrantVectorStore.from_documents(
        documents=new_splits,
        embedding=embeddings,
        location=":memory:",
        collection_name=f"tune_{label}",
        force_recreate=True,
    )

    scored = store.similarity_search_with_score(question, k=k)
    scores = [s for _, s in scored]

    context = format_context(scored)
    answer = rag_chain.invoke({"context": context, "question": question})

    top_doc, top_score = scored[0]
    top_page = top_doc.metadata.get("page")
    top_page_display = top_page + 1 if isinstance(top_page, int) else "?"
    top_preview = top_doc.page_content[:220].replace("\n", " ")

    print(f"\n=== {label} (chunk={chunk_size}, overlap={chunk_overlap}, k={k}) ===")
    print(f"Chunks produced: {len(new_splits)}")
    print(f"Top score: {max(scores):.3f}   Mean top-{k} score: {sum(scores)/len(scores):.3f}")
    print(f"Top hit (page {top_page_display}): {top_preview}")
    print(f"\nAnswer:\n{answer}")
    return {"label": label, "scores": scores, "answer": answer, "n_chunks": len(new_splits)}


results = [run_experiment(question=test_question, **cfg) for cfg in experiments]

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
for r in results:
    top = max(r["scores"])
    mean = sum(r["scores"]) / len(r["scores"])
    print(f"{r['label']:22s}  n_chunks={r['n_chunks']:4d}  top={top:.3f}  mean={mean:.3f}")


### 🏗️ Activity Notes

- **Setting changed:** I compared three configurations on the same test question (*"What signs suggest that a cat should be seen by a veterinarian?"*):
  - Baseline: `chunk_size=1000`, `chunk_overlap=200`, `k=4`
  - A — smaller chunks: `chunk_size=400`, `chunk_overlap=80`, `k=4`
  - B — wider retrieval: `chunk_size=1000`, `chunk_overlap=200`, `k=8`

- **Before:** With the baseline I got 135 chunks. Top similarity score was 0.584, mean across top-4 was 0.568. The top hit landed on page 8 (the section about pain/anxiety and quality of life). The generated answer was a broad practical list — vomiting, appetite changes, drinking/urination, mobility, grooming, anxiety.

- **After:**
  - Variant A produced 345 chunks (smaller chunks → more of them). Top score dropped slightly to 0.570 and mean to 0.566. The top hit shifted to page 10, which is more senior-cat-specific content, and the generated answer became narrower and mostly focused on senior cats.
  - Variant B kept the same 135 chunks as baseline, so the top hit and top score didn't change (still 0.584). But because I asked for 8 chunks instead of 4, the mean dropped to 0.556 — chunks 5–8 were noticeably less relevant than the first four.

- **Did retrieval improve? Why or why not?** No — the baseline was the best of the three. My initial intuition was that smaller chunks would give more precise matches, but the cat health guideline is already structured into topical paragraphs, and cutting them smaller fragmented those units. The "best match" then shifted to a more narrow section (senior cats) instead of the broader pain/quality-of-life section that actually answers the question better. Larger `k` didn't help either — it just diluted the average score with weaker matches without changing the top hit. Takeaway: there's no universally "better" setting; the right tuning depends on how the source document is structured and how specific the query is.
